# Sacred Fruit: revisar datasets candidatos

Este notebook **no entrena nada**. Baja cada dataset candidato de Roboflow Universe y muestra:

- licencia, tipo de proyecto (tiene que ser *instance segmentation*) y cantidad de imágenes;
- las clases originales con cuántas etiquetas tiene cada una;
- 3 fotos de ejemplo con los polígonos dibujados y el nombre de la clase.

Con eso se decide qué datasets aprobar antes de sumarlos a `sacred_fruit_reentrenamiento.ipynb`. Alcanza con CPU.

In [ ]:
!pip -q install roboflow

In [ ]:
import getpass
ROBOFLOW_API_KEY = globals().get("ROBOFLOW_API_KEY") or getpass.getpass("Pegá tu Private API Key de Roboflow y apretá Enter: ")

# (workspace, proyecto, para qué sección)
CANDIDATES = [
    ("machine-learning-fruits", "orange-rotting-segmentation", "4.1 defecto / 4.2 naranja sobrepasada"),
    ("achuka", "mnmnmnmn-rcrxr", "4.1 defecto en mango"),
    ("dania-psle4", "fruit-disease-quglp", "4.1 defecto"),
    ("sansan-5p4nm", "fruit-seg1", "4.2 mango sobrepasado"),
    ("tomatolabel", "tomato-detect-jyqme", "4.2 tomate sobrepasado"),
    ("guerin-agossadou-knqcx", "tomatoes2-inzgt", "4.2 tomate"),
    ("rutuja-xh9en", "fruit-and-vegetable-detection-psq4v-a6u5j", "4.5 otras frutas como fondo"),
    ("roboflow-hcfp3", "fruits-detection-k9rcf", "4.5 otras frutas como fondo"),
    ("fruit-ripening", "banana-ripening-process", "4.3 banana"),
]

In [ ]:
import json, glob, random, urllib.request, yaml
from collections import Counter
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
random.seed(0)
COLORS = ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4', '#46f0f0', '#f032e6', '#bcf60c', '#fabebe', '#008080']

def info(ws, pj):
    url = f"https://api.roboflow.com/{ws}/{pj}?api_key={ROBOFLOW_API_KEY}"
    data = json.load(urllib.request.urlopen(url))
    p = data.get("project", data)
    return {k: p.get(k) for k in ("name", "type", "license", "images", "annotation", "classes")}, data.get("versions", [])

def draw(img_path, lbl_path, names):
    im = Image.open(img_path).convert("RGB"); w, h = im.size
    d = ImageDraw.Draw(im, "RGBA")
    if Path(lbl_path).exists():
        for ln in open(lbl_path).read().splitlines():
            p = ln.split()
            if len(p) < 7: continue
            c = int(p[0]); pts = [(float(x) * w, float(y) * h) for x, y in zip(p[1::2], p[2::2])]
            col = COLORS[c % len(COLORS)]
            d.polygon(pts, outline=col, fill=col + "40", width=3)
            d.text(pts[0], names[c], fill=col)
    return im

report = []
for ws, pj, why in CANDIDATES:
    print("=" * 90, f"\n{ws}/{pj}  ({why})\nhttps://universe.roboflow.com/{ws}/{pj}")
    try:
        meta, versions = info(ws, pj)
        print("Datos del proyecto:", meta)
        p = rf.workspace(ws).project(pj)
        v = max(int(str(x.version).split("/")[-1]) for x in p.versions())
        d = p.version(v).download("yolov8", location=f"/content/check/{pj}", overwrite=False)
        names = yaml.safe_load(open(Path(d.location) / "data.yaml"))["names"]
        names = names if isinstance(names, list) else [names[k] for k in sorted(names)]
        imgs = glob.glob(f"{d.location}/*/images/*")
        cnt, polys, boxes = Counter(), 0, 0
        for img in imgs:
            lbl = Path(img.replace("/images/", "/labels/")).with_suffix(".txt")
            if lbl.exists():
                for ln in open(lbl).read().splitlines():
                    q = ln.split()
                    if not q: continue
                    cnt[names[int(q[0])]] += 1
                    polys += len(q) >= 7; boxes += len(q) == 5
        print(f"Versión {v} | imágenes: {len(imgs)} | polígonos: {polys} | cajas: {boxes}")
        print("Etiquetas por clase:", dict(cnt))
        report.append((ws, pj, why, meta.get("license"), len(imgs), dict(cnt), polys, boxes))
        sample = random.sample(imgs, min(3, len(imgs)))
        fig, axes = plt.subplots(1, len(sample), figsize=(6 * len(sample), 6))
        for ax, img in zip(axes if len(sample) > 1 else [axes], sample):
            ax.imshow(draw(img, Path(img.replace("/images/", "/labels/")).with_suffix(".txt"), names)); ax.axis("off")
        plt.suptitle(f"{ws}/{pj}"); plt.show()
    except Exception as e:
        print("No se pudo revisar:", e)

In [ ]:
print("| Dataset | Para | Licencia | Imágenes | Polígonos | Cajas | Clases (etiquetas) |")
print("| --- | --- | --- | --- | --- | --- | --- |")
for ws, pj, why, lic, n, cnt, polys, boxes in report:
    print(f"| [{ws}/{pj}](https://universe.roboflow.com/{ws}/{pj}) | {why} | {lic} | {n} | {polys} | {boxes} | {cnt} |")